In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

try:
    import xgboost as xgb
except:
    !pip install "xgboost<3"
    import xgboost as xgb

#### Constants

In [ ]:
str_bucket = 'assessment-alt'
str_task = '06_model_eval'
str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Bucket: {str_bucket}')
print(f'Task: {str_task}')

#### Load Data

In [ ]:
%%time
# Load test predictions
df_pred = pd.read_csv(f's3://{str_bucket}/05_model/test_predictions.csv')
print(f'Predictions: {df_pred.shape}')

# Load X_test for feature analysis
X_test = pd.read_csv(f's3://{str_bucket}/04_preprocessing/X_test.csv')
print(f'X_test: {X_test.shape}')

# Load model
model = xgb.Booster()
model.load_model('../05_model/output/xgb_model.json')
print('Model loaded')

df_pred.head()

#### Define Metrics

In [ ]:
def calc_metrics(arr_true, arr_pred):
    """Calculate regression metrics on actual price scale."""
    arr_true = np.array(arr_true, dtype=float)
    arr_pred = np.array(arr_pred, dtype=float)
    flt_mae = np.mean(np.abs(arr_true - arr_pred))
    flt_mape = np.mean(np.abs((arr_true - arr_pred) / np.clip(arr_true, 1, None))) * 100
    flt_rmse = np.sqrt(np.mean((arr_true - arr_pred) ** 2))
    flt_median_ae = np.median(np.abs(arr_true - arr_pred))
    flt_r2 = 1 - np.sum((arr_true - arr_pred)**2) / np.sum((arr_true - np.mean(arr_true))**2)
    return {
        'MAE': round(flt_mae, 2),
        'MAPE (%)': round(flt_mape, 2),
        'RMSE': round(flt_rmse, 2),
        'Median AE': round(flt_median_ae, 2),
        'R2': round(flt_r2, 4),
    }

#### Model Metrics

In [ ]:
dict_model_metrics = calc_metrics(df_pred['flt_price_true'], df_pred['flt_price_pred'])
print('Model Metrics (actual price scale):')
for str_key, val in dict_model_metrics.items():
    print(f'  {str_key}: {val}')

#### Compare to Baseline

In [ ]:
try:
    df_baseline = pd.read_csv('../02_price_estimator/output/baseline_metrics.csv')
    print('Baseline Metrics:')
    print(df_baseline.to_string(index=False))
    print(f'\nModel Metrics:')
    print(pd.DataFrame([dict_model_metrics]).to_string(index=False))
except FileNotFoundError:
    print('Baseline metrics not found — run 02_price_estimator first')
    print(f'\nModel Metrics:')
    print(pd.DataFrame([dict_model_metrics]).to_string(index=False))

#### Actual vs Predicted

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df_pred['flt_price_true'], df_pred['flt_price_pred'], alpha=0.1, s=5, color='steelblue')
flt_min = max(df_pred['flt_price_true'].min(), df_pred['flt_price_pred'].min(), 1)
flt_max = max(df_pred['flt_price_true'].max(), df_pred['flt_price_pred'].max())
ax.plot([flt_min, flt_max], [flt_min, flt_max], 'r--', linewidth=2, label='Perfect')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Actual vs Predicted Price')
ax.set_xlabel('Actual Price ($)')
ax.set_ylabel('Predicted Price ($)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

#### Residual Analysis

In [ ]:
arr_residuals = df_pred['flt_price_true'] - df_pred['flt_price_pred']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Residual distribution
axes[0].hist(np.clip(arr_residuals, -1000, 1000), bins=100, color='steelblue', edgecolor='black')
axes[0].set_title('Residual Distribution (clipped +/- $1000)')
axes[0].set_xlabel('Residual ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(0, color='red', linestyle='--')

# Residuals vs predicted
axes[1].scatter(df_pred['flt_price_pred'], arr_residuals, alpha=0.1, s=5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xscale('log')
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

#### Error by Price Range

In [ ]:
list_bins = [0, 50, 100, 250, 500, 1000, float('inf')]
list_labels = ['$0-50', '$50-100', '$100-250', '$250-500', '$500-1K', '$1K+']
df_pred['str_price_bin'] = pd.cut(df_pred['flt_price_true'], bins=list_bins, labels=list_labels)

list_dict_bin_metrics = []
for str_bin in list_labels:
    df_bin = df_pred[df_pred['str_price_bin'] == str_bin]
    if len(df_bin) > 0:
        dict_bin = calc_metrics(df_bin['flt_price_true'], df_bin['flt_price_pred'])
        dict_bin['Price Range'] = str_bin
        dict_bin['N'] = len(df_bin)
        list_dict_bin_metrics.append(dict_bin)

df_bin_metrics = pd.DataFrame(list_dict_bin_metrics)
df_bin_metrics = df_bin_metrics[['Price Range', 'N', 'MAE', 'MAPE (%)', 'RMSE', 'Median AE', 'R2']]
print(df_bin_metrics.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(df_bin_metrics['Price Range'], df_bin_metrics['MAPE (%)'], color='steelblue', edgecolor='black')
ax.set_title('MAPE by Price Range')
ax.set_xlabel('Price Range')
ax.set_ylabel('MAPE (%)')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/error_by_price_range.png', dpi=150, bbox_inches='tight')
plt.show()

#### Error by Grade

In [ ]:
df_pred['flt_grade'] = X_test['flt_grade'].values

list_dict_grade_metrics = []
for flt_grade in sorted(df_pred['flt_grade'].unique()):
    df_grade = df_pred[df_pred['flt_grade'] == flt_grade]
    if len(df_grade) >= 10:
        dict_grade = calc_metrics(df_grade['flt_price_true'], df_grade['flt_price_pred'])
        dict_grade['Grade'] = flt_grade
        dict_grade['N'] = len(df_grade)
        list_dict_grade_metrics.append(dict_grade)

df_grade_metrics = pd.DataFrame(list_dict_grade_metrics)
print(df_grade_metrics[['Grade', 'N', 'MAE', 'MAPE (%)', 'Median AE']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([str(g) for g in df_grade_metrics['Grade']], df_grade_metrics['MAPE (%)'],
       color='steelblue', edgecolor='black')
ax.set_title('MAPE by Grade')
ax.set_xlabel('Grade')
ax.set_ylabel('MAPE (%)')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/error_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

#### Feature Importance

In [ ]:
dict_importance = model.get_score(importance_type='gain')
df_importance = pd.DataFrame({
    'Feature': list(dict_importance.keys()),
    'Gain': list(dict_importance.values()),
}).sort_values('Gain', ascending=False)

print(df_importance.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
df_importance_plot = df_importance.sort_values('Gain', ascending=True)
ax.barh(df_importance_plot['Feature'], df_importance_plot['Gain'], color='steelblue', edgecolor='black')
ax.set_title('Feature Importance (Gain)')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

#### Prediction Coverage — Seen vs Unseen Cards

In [ ]:
# Cards with 0 card_txn_count are unseen in training
df_pred['int_card_txn_count'] = X_test['int_card_txn_count'].values
df_seen = df_pred[df_pred['int_card_txn_count'] > 0]
df_unseen = df_pred[df_pred['int_card_txn_count'] == 0]

print(f'Seen cards: {len(df_seen):,} ({len(df_seen)/len(df_pred)*100:.1f}%)')
print(f'Unseen cards: {len(df_unseen):,} ({len(df_unseen)/len(df_pred)*100:.1f}%)')

if len(df_seen) > 0:
    print(f'\nSeen card metrics:')
    dict_seen = calc_metrics(df_seen['flt_price_true'], df_seen['flt_price_pred'])
    for str_key, val in dict_seen.items():
        print(f'  {str_key}: {val}')

if len(df_unseen) > 0:
    print(f'\nUnseen card metrics:')
    dict_unseen = calc_metrics(df_unseen['flt_price_true'], df_unseen['flt_price_pred'])
    for str_key, val in dict_unseen.items():
        print(f'  {str_key}: {val}')

#### Save Metrics

In [ ]:
dict_all_metrics = {
    'model_metrics': dict_model_metrics,
    'seen_card_metrics': dict_seen if len(df_seen) > 0 else {},
    'unseen_card_metrics': dict_unseen if len(df_unseen) > 0 else {},
}
with open(f'{str_dirname_output}/model_metrics.json', 'w') as f:
    json.dump(dict_all_metrics, f, indent=2)
print(f'Saved model_metrics.json to {str_dirname_output}')

#### Takeaways

- **Model performance vs baseline**: Compare MAE, MAPE, and R² to the non-ML baselines from 02_price_estimator. The XGBoost model should improve on the combined baseline, especially for cards with limited transaction history.
- **Price range sensitivity**: Low-value cards ($0-50) tend to have higher MAPE since small dollar errors become large percentage errors. High-value cards ($1K+) have larger absolute errors but are harder to predict due to rarity.
- **Feature importance**: Card-level median/mean price features likely dominate — the model learns that historical price is the strongest predictor. Subject and brand stats provide fallback signal for unseen cards.
- **Seen vs unseen cards**: The model performs significantly better on cards seen in training. For truly novel cards, predictions rely on subject/brand/grade features.
- **Improvements to consider**:
  - More granular time features (day of week, market momentum)
  - Variety-level statistics
  - Player performance / award features (external data)
  - Ensemble with the non-ML baseline
  - Quantile regression for confidence intervals